In [ ]:
import os

# Save the current PATH
original_path = os.environ['PATH']

# Set CUDA 12.5 environment variables, appending the original PATH explicitly
os.environ['CUDA_HOME'] = '/usr/local/cuda-12.5'
os.environ['PATH'] = f"/usr/local/cuda-12.5/bin:{original_path}"
os.environ['LD_LIBRARY_PATH'] = f"/usr/local/cuda-12.5/lib64:{os.environ.get('LD_LIBRARY_PATH', '')}"

#!rm -rf /home/ids/yuhe/.cache/torch_extensions
CODE_DIR = '/home/ids/yuhe/Projects/CA_with_GAN/3_code/styleGAN/pSp_encoder_constructive'

import os
os.chdir(f'{CODE_DIR}')

notebook_path = os.getcwd()
print('Current working directory is:', '\n', notebook_path) 

# from argparse import Namespace
# import time
# import sys
# import pprint
# import numpy as np
# from PIL import Image
# import torch
# import torchvision.transforms as transforms
# import matplotlib.pyplot as plt
# from IPython.display import display

# sys.path.append(".")
# sys.path.append("..")

# # from datasets import augmentations
# from utils.common import tensor2im, log_input_image
# # from models.psp import pSp

# from notebooks.def_funcs import load_sparsity_model, evaluate_model, transform_images_to_batch, load_folder_images, \
#     show_latent_map, visulize_singleImg_paired2, visulize_singleImg_paired3, visulize_singleImg_paired4, visulize_singleImg_paired5
# %load_ext autoreload
# %autoreload 2

In [96]:
import torch
from sklearn.decomposition import PCA

def compute_PCA_weighted_loss(from_latent, on_latent, norm_type="exp", weight_axis="columns", scale_factor=10):
    """
    Compute the weighted loss || weights * from_latent - weights * on_latent ||^2 using weights derived
    independently for each sample in the batch with a choice of normalization.

    Parameters:
    - from_latent (torch.Tensor): Reference tensor of shape (batch_size, 18, 512).
    - on_latent (torch.Tensor): Target tensor of shape (batch_size, 18, 512).
    - norm_type (str): Normalization method ('exp' for exponential or 'l2' for L2 norm).
    - weight_axis (str): Apply weights along "columns" (features) or "rows" (samples).

    Returns:
    - weighted_on_latent (torch.Tensor): Weighted on_latent tensor.
    - normalized_weights (torch.Tensor): Normalized weights for each sample.
    """
    batch_size, num_rows, num_features = from_latent.shape

    # Placeholder for weights and weighted_latent
    normalized_weights = []
    weighted_latent = torch.zeros_like(on_latent, device=on_latent.device)

    for i in range(batch_size):
        # Extract single sample
        sample_from_latent = from_latent[i].cpu().numpy()
        
        # Perform PCA on a single sample
        if weight_axis == "columns":
            pca = PCA(n_components=1)
            pca.fit(sample_from_latent)  # Shape: (18, 512)
            first_component = torch.tensor(pca.components_[0], dtype=torch.float32, device=from_latent.device)  # Shape: (512,)
        elif weight_axis == "rows":
            pca = PCA(n_components=1)
            pca.fit(sample_from_latent.T)  # Shape: (512, 18)
            first_component = torch.tensor(pca.components_[0], dtype=torch.float32, device=from_latent.device)  # Shape: (18,)
        else:
            raise ValueError("Invalid weight_axis. Choose 'columns' or 'rows'.")

        # Normalize weights
        if norm_type == "exp":
            exp_weights = torch.exp(first_component)
            sample_weights = exp_weights / torch.sum(exp_weights)
        elif norm_type == "l2":
            l2_weights = torch.abs(first_component)
            sample_weights = l2_weights / torch.sqrt(torch.sum(l2_weights**2))
        else:
            raise ValueError("Invalid norm_type. Choose 'exp' or 'l2'.")

        # Apply weights to on_latent for this sample
        if weight_axis == "columns":
            weighted_latent[i] = on_latent[i] * sample_weights.view(1, -1)  # Shape: (18, 512)
        elif weight_axis == "rows":
            weighted_latent[i] = on_latent[i] * sample_weights.view(-1, 1)  # Shape: (18, 512)

        # # Store weights
        # normalized_weights.append(sample_weights)

    # # Stack weights for batch
    # normalized_weights = torch.stack(normalized_weights, dim=0)

    return weighted_latent * scale_factor #, normalized_weights


In [76]:
# Example batch of latent tensors
batch_size = 4
from_latent = torch.rand(batch_size, 18, 512)  # Reference tensor
on_latent = torch.rand(batch_size, 18, 512)    # Target tensor

# Compute weighted latent tensor using column-wise weighting with exponential normalization
weighted_latent_columns, normalized_weights_columns = compute_PCA_weighted_loss(
    from_latent, on_latent, norm_type="exp", weight_axis="columns"
)

print("Column-wise Weighted Latent Shape:", weighted_latent_columns.shape)
print("Normalized Weights (Columns) Shape:", normalized_weights_columns.shape)

# Compute weighted latent tensor using row-wise weighting with L2 normalization
weighted_latent_rows, normalized_weights_rows = compute_PCA_weighted_loss(
    from_latent, on_latent, norm_type="l2", weight_axis="rows"
)

print("Row-wise Weighted Latent Shape:", weighted_latent_rows.shape)
print("Normalized Weights (Rows) Shape:", normalized_weights_rows.shape)


Column-wise Weighted Latent Shape: torch.Size([4, 18, 512])
Normalized Weights (Columns) Shape: torch.Size([4, 512])
Row-wise Weighted Latent Shape: torch.Size([4, 18, 512])
Normalized Weights (Rows) Shape: torch.Size([4, 18])


In [ ]:
# Example batch of latent tensors
batch_size = 4
from_latent = torch.rand(batch_size, 18, 512)  # Reference tensor
on_latent = torch.rand(batch_size, 18, 512)    # Target tensor
on_latent2 = torch.rand(batch_size, 18, 512)    # Target tensor

weighted_latent1 = compute_PCA_weighted_loss(
    from_latent, on_latent, norm_type="exp", weight_axis="rows", scale_factor=10)


weighted_latent2 = compute_PCA_weighted_loss(
    from_latent, on_latent2, norm_type="exp", weight_axis="rows", scale_factor=10)

import torch.nn.functional as F
a = F.mse_loss(weighted_latent1, weighted_latent2)
b = F.mse_loss(on_latent, on_latent2)
print(a)
print(b)




tensor(5.3719)
tensor(0.1649)


tensor(9213.9805)
tensor(0.1665)
